#### Cell 1: Set up Environment

In [ ]:
# ==========================================
# CELL 1: RAW INSTALLATION & DEPENDENCY FIX
# ==========================================
from IPython.display import Javascript, display
def resize_colab_cell():
  display(Javascript('google.colab.output.setIframeHeight(0, true, {maxHeight: 20000})'))
get_ipython().events.register('pre_run_cell', resize_colab_cell)
from google.colab import output
output.no_vertical_scroll()

%cd /content
!rm -rf DeepKG
!git clone https://github.com/DSC-SPIDAL/DeepKG.git > /dev/null 2>&1

# 🔥 FIX: Uninstall deprecated packages and force the new GenAI SDK
!pip uninstall -y google-generativeai llama-index-llms-gemini llama-index-embeddings-gemini > /dev/null 2>&1
!pip install -qU "google-genai>=2.0.0" openai "anthropic[vertex]" llama-index llama-index-llms-google-genai llama-index-embeddings-google-genai llama-index-retrievers-bm25 gspread bm25s python-dotenv nest_asyncio arxiv pypdf google-api-python-client > /dev/null 2>&1

print("✅ Dependencies Installed. STOP -> Go to Runtime > Restart Session, then run Cell 2.")

<IPython.core.display.Javascript object>

/content
✅ Dependencies Installed. STOP -> Go to Runtime > Restart Session, then run Cell 2.


#### Cell 2: Authenticate

In [ ]:
# ==========================================
# CELL 2: AUTHENTICATION & SECRETS
# ==========================================
from IPython.display import Javascript, display
def resize_colab_cell():
  display(Javascript('google.colab.output.setIframeHeight(0, true, {maxHeight: 20000})'))
get_ipython().events.register('pre_run_cell', resize_colab_cell)
from google.colab import output
output.no_vertical_scroll()

import os
from google.colab import auth, userdata, drive

print("🔑 1. Authenticating with Google...")
auth.authenticate_user()

print("📂 2. Mounting Drive...")
drive.mount('/content/drive', force_remount=True)

print("🔒 3. Loading Secrets securely to OS Environment...")
def lock_secret(secret_name):
    try:
        val = userdata.get(secret_name)
        if val: os.environ[secret_name] = val
    except: pass

lock_secret('GEMINI_API_KEY')
# Ensure old legacy key format doesn't conflict with new SDK
if "GOOGLE_API_KEY" in os.environ: del os.environ["GOOGLE_API_KEY"]

lock_secret('OPENAI_API_KEY')
lock_secret('ANTHROPIC_API_KEY')
lock_secret('GOOGLE_CLOUD_ID')
lock_secret('GCP_PROJECT_ID')
lock_secret('DRIVE_BENCH_ID')
lock_secret('DRIVE_LOG_FOLDER_ID')
lock_secret('KB_SHEET_ID')
lock_secret('PROJECT_LIST_ID')

%cd /content/DeepKG
print("✅ Environment Ready. Proceed to Cell 3.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

🔑 1. Authenticating with Google...
📂 2. Mounting Drive...
Mounted at /content/drive
🔒 3. Loading Secrets securely to OS Environment...
/content/DeepKG
✅ Environment Ready. Proceed to Cell 3.


#### 3:Execute

In [ ]:
# ==========================================
# CELL 3: RUN THE TRUE CLOUD BASELINE
# ==========================================
from IPython.display import Javascript, display
def resize_colab_cell():
  display(Javascript('google.colab.output.setIframeHeight(0, true, {maxHeight: 20000})'))
get_ipython().events.register('pre_run_cell', resize_colab_cell)
from google.colab import output
output.no_vertical_scroll()

import os, glob, warnings, nest_asyncio, re, io, time, asyncio, requests, importlib
import pandas as pd
from google.colab import userdata, auth
import gspread
from google.auth import default

warnings.filterwarnings("ignore", category=FutureWarning)
nest_asyncio.apply()

def get_secret(secret_name, fallback_value=""):
    try: return os.environ.get(secret_name) or userdata.get(secret_name) or fallback_value
    except: return fallback_value

# ---------------------------------------------------------
# 🎯 EXECUTION TARGETS
# ---------------------------------------------------------
PROJECT_NAMES = [ "UTSD", "TIMEBENCH", "LOTSA", "TEMPO", "TSFM", "KAGGLETS", "M2", "M6"]
TARGET_RUN = "Run3"

# 👉 The Backbone Brain for Deep Research & Planning
GEMINI_SEARCH_MODEL = "gemini-3.1-pro-preview"

# 👉 CHOOSE YOUR RAG PROVIDER
TARGET_RAG_PROVIDER = "GEMINI"

# 👉 CHOOSE YOUR RAG MODEL (e.g., gemini-3.1-pro-preview gemini-3.7-flash gemini-3.6-flash, gemini-3.5-flash-lite, gemini-3.5-flash)
TARGET_RAG_MODEL = "gemini-3.8-flash"

# 👉 DYNAMIC SECRET FETCH FOR GCP PROJECT ID (Used only for Vertex AI)
GCP_PROJECT_ID = get_secret("GOOGLE_CLOUD_ID") or get_secret("GCP_PROJECT_ID")
VERTEX_REGION = "us-east5"

os.environ["TARGET_RUN"] = TARGET_RUN
os.environ["TARGET_PROVIDER"] = TARGET_RAG_PROVIDER
os.environ["TARGET_MODEL"] = TARGET_RAG_MODEL
os.environ["TARGET_PRO_MODEL"] = GEMINI_SEARCH_MODEL
os.environ["TARGET_FLASH_MODEL"] = TARGET_RAG_MODEL
os.environ["VERTEX_REGION"] = VERTEX_REGION
if GCP_PROJECT_ID: os.environ["GCP_PROJECT_ID"] = GCP_PROJECT_ID

# 🔥 Enable the GitHub-native Amnesia, Benchmark, and Export parameters
os.environ["BENCHMARK_MODE"] = "CLOUD"
os.environ["APPLY_AMNESIA"] = "True"
os.environ["DRIVE_BENCH_ID"] = get_secret("DRIVE_BENCH_ID")

# ---------------------------------------------------------
# 🔧 ENVIRONMENT ENFORCEMENT & FIXES
# ---------------------------------------------------------
print(f"🔧 Patching DeepCollector Codebase for {TARGET_RAG_MODEL}...")
for filepath in glob.glob("/content/DeepKG/**/*.py", recursive=True):
    with open(filepath, "r") as f: content = f.read()
    new_content = content.replace("llama_index.llms.gemini", "llama_index.llms.google_genai")
    new_content = new_content.replace("from llama_index.llms.google_genai import Gemini", "from llama_index.llms.google_genai import GoogleGenAI as Gemini")
    new_content = new_content.replace("llama_index.embeddings.gemini", "llama_index.embeddings.google_genai")
    new_content = new_content.replace("from llama_index.embeddings.google_genai import GeminiEmbedding", "from llama_index.embeddings.google_genai import GoogleGenAIEmbedding as GeminiEmbedding")
    if new_content != content:
        with open(filepath, "w") as f: f.write(new_content)

# Ensure modules are loaded cleanly to avoid recursion and attribute errors
import sys
for mod in list(sys.modules.keys()):
    if mod.startswith('deepcollector.'):
        del sys.modules[mod]

# Move to the correct directory so local imports resolve correctly
if os.path.exists("/content/DeepKG"):
    os.chdir("/content/DeepKG")

import deepcollector.tools.research
importlib.reload(deepcollector.tools.research)
from deepcollector.tools.research import ResearchTools

# ---------------------------------------------------------
# 🚀 EXECUTION
# ---------------------------------------------------------
print("\n🔑 Acquiring Credentials for Job...")
creds, _ = default()
gc = gspread.authorize(creds)

from deepcollector.config.settings import AppConfig
from deepcollector.core.executor import execute_jobs

# Pass the cached keys natively
config = AppConfig(
    VERBOSITY_LEVEL=1, SECRETS={"GEMINI_API_KEY": get_secret("GEMINI_API_KEY", "")},
    GOOGLE_SHEET_KB_INPUT = get_secret("KB_SHEET_ID"),
    GOOGLE_SHEET_PROJECT_LIST_INPUT = get_secret("PROJECT_LIST_ID"),
    GOOGLE_DRIVE_SHEET_FOLDER_ID = get_secret("DRIVE_BENCH_ID"),
    GOOGLE_DRIVE_LOG_FOLDER_ID = get_secret("DRIVE_LOG_FOLDER_ID"),
    ENABLE_DEEP_RESEARCH=True, ENABLE_GOLDEN_FASTPATH=False,
    ENABLE_PREFLIGHT_CRAWLER=True, ENABLE_ARBITRATION_PROMPT=True, ENABLE_STRICT_TAXONOMY=True,
    ENABLE_MULTI_QUERY_RAG=True, ENABLE_VARIANT_MAPPING=True, ENABLE_SINGLETON_VERIFICATION=True, ENABLE_ORACLE_SEARCH=True
)

if TARGET_RAG_PROVIDER != "GEMINI":
    config.CELLULAR_RAG_BATCH_SIZE = 5
else:
    config.CELLULAR_RAG_BATCH_SIZE = 10

os.environ["DEEPCOLLECTOR_LOG_FOLDER_ID"] = config.GOOGLE_DRIVE_LOG_FOLDER_ID or ""
config.recalculate_runtime_parameters()
config._process_sheet_ids()

print(f"\n🚀 Firing HYBRID CLOUD RUN ({TARGET_RAG_PROVIDER}: {TARGET_RAG_MODEL} | {TARGET_RUN}) for {PROJECT_NAMES}...")
try:
    execute_jobs(mode="AGENT", project_names=PROJECT_NAMES, base_config=config, gc_client=gc, dry_run=False)
except Exception as e:
    print(f"\n❌ Main Execution Failed or Aborted: {e}")